# Buổi 3: Hopfield khử nhiễu chữ C / I / T

Nối tiếp buổi 2: cùng ba mẫu chữ **C, I, T**, nhưng đổi từ lưới 3×3 sang lưới **4×4** (16 ô) để khớp với bài giảng buổi 3.

**Thesis:** Hamming (buổi 2) dán nhãn — trả về "gần C nhất" nhưng không sửa ô nào. Hopfield (buổi 3) vá hình — sửa từng pixel cho tới khi ảnh trở về đúng mẫu đã nhớ.

Ba phần chính:
1. **Ôn Hamming trên lưới 4×4** — cùng dữ liệu sẽ dùng suốt buổi.
2. **Luật Hebb** — dựng ma trận trọng số `W` (16×16) từ ba mẫu.
3. **Hopfield khử nhiễu** — cập nhật `xᵢ ← sign(netᵢ)` cho tới khi ảnh sạch, kèm theo dõi năng lượng `E`.

Cách học: chạy lần lượt, mỗi phần đều có phép tính tay rồi kiểm tra bằng code.

---
## Phần 0: Đánh số ô và ba mẫu chuẩn C, I, T (lưới 4×4)

Quy ước: quét trái → phải, trên → dưới. Ô thứ `i = 4(hàng−1) + cột`. Đen = `+1`, trắng = `−1`.

```
1  2  3  4
5  6  7  8
9  10 11 12
13 14 15 16
```

Cùng thứ tự này dùng cho Hamming, luật Hebb và cập nhật Hopfield.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Ba mẫu chuẩn trên lưới 4x4 (bipolar: +1 = đen, -1 = trắng)
patterns = {
    'C': np.array([ 1, 1, 1, 1,
                    1,-1,-1,-1,
                    1,-1,-1,-1,
                    1, 1, 1, 1], dtype=float),
    'I': np.array([-1, 1, 1,-1,
                   -1, 1, 1,-1,
                   -1, 1, 1,-1,
                   -1, 1, 1,-1], dtype=float),
    'T': np.array([ 1, 1, 1, 1,
                   -1, 1, 1,-1,
                   -1, 1, 1,-1,
                   -1, 1, 1,-1], dtype=float),
}

def show_grid(ax, vec, title):
    grid = vec.reshape(4, 4)
    ax.imshow(grid, cmap='gray_r', vmin=-1, vmax=1)
    for i in range(4):
        for j in range(4):
            val = grid[i, j]
            color = 'white' if val == 1 else 'black'
            ax.text(j, i, f'{int(val):+d}', ha='center', va='center',
                    fontsize=9, color=color, fontweight='bold')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xticks([])
    ax.set_yticks([])

fig, axes = plt.subplots(1, 3, figsize=(9, 3.2))
for ax, (name, p) in zip(axes, patterns.items()):
    show_grid(ax, p, f'Mẫu {name}')
plt.suptitle('Ba mẫu chuẩn trên lưới 4×4 — mã hóa bipolar', fontsize=13)
plt.tight_layout()
plt.show()

print("Vector mẫu (16 chiều):")
for name, p in patterns.items():
    print(f"  xi{name} = {p.astype(int).tolist()}")

---
## Phần 1: Ôn Hamming — input C lật 2 pixel

Input nhiễu là chữ C lật ô 4 (H1C4: +1→−1) và ô 13 (H4C1: +1→−1).

**Công thức:** `yₖ = ξₖᵀ·x = N − 2d` (N=16, d = số bit lệch).

Hamming tính một lần rồi chọn điểm cao nhất — **không sửa ô nào sai**.

In [ ]:
# Input nhiễu: chữ C lật ô 4 và ô 13 (chỉ số 0-based: 3 và 12)
x0 = patterns['C'].copy()
x0[3] *= -1   # ô 4
x0[12] *= -1  # ô 13

P = np.array([patterns['C'], patterns['I'], patterns['T']])
names = ['C', 'I', 'T']

scores = P @ x0

print("Input x(0): chữ C lật ô 4 và ô 13")
print(f"x(0) = {x0.astype(int).tolist()}\n")

print(f"{'Mẫu':>5} | {'y_k':>5} | {'d (bit lệch)':>13}")
print("-" * 32)
for name, score in zip(names, scores):
    d = (16 - score) / 2
    print(f"  {name}   | {int(score):>4}  |    {int(d)}/16")

winner = names[np.argmax(scores)]
print(f"\n→ Kết luận Hamming: gần {winner} nhất (điểm {int(scores.max())}).")
print("   Nhưng 2 ô (4 và 13) trên lưới vẫn còn sai — Hamming không vá được.")

fig, ax = plt.subplots(1, 1, figsize=(3.2, 3.2))
show_grid(ax, x0, f'x(0) → Hamming: "{winner}"')
plt.show()

**Câu chuyển sang Hopfield:** Máy đã biết là C rồi — làm sao vá 2 ô mờ? Cần một mạng có thể **đổi trạng thái từng ô**.

---
## Phần 2: Luật Hebb — dựng ma trận W 16×16

**Công thức (dạng scalar):**
```
wᵢⱼ = Σ_μ ξᵢ^μ · ξⱼ^μ   (μ = C, I, T),   wᵢᵢ = 0
```

**Dạng ma trận (để code):**
```
W = Σ_μ ξ^μ (ξ^μ)ᵀ,   diag(W) = 0
```

Mỗi tích `ξᵢ·ξⱼ` là **+1** nếu hai ô cùng màu trên mẫu đó, **−1** nếu trái màu. Với 3 mẫu, `w ∈ {−3, −1, +1, +3}`.

In [ ]:
xiC, xiI, xiT = patterns['C'], patterns['I'], patterns['T']

W = np.outer(xiC, xiC) + np.outer(xiI, xiI) + np.outer(xiT, xiT)
np.fill_diagonal(W, 0)

print("Ma trận W (16×16), vài dòng đầu:")
print(W[:4, :8])

# Đối chiếu tay với ba cặp ô mẫu trong bài giảng
print("\nĐối chiếu tay (chỉ số 1-based -> 0-based):")
checks = [(1, 4, 3), (5, 6, -3), (1, 2, 1)]
for i, j, expected in checks:
    w_ij = W[i-1, j-1]
    status = "OK" if w_ij == expected else "SAI"
    print(f"  w_{i},{j} = {int(w_ij):+d}  (kỳ vọng {expected:+d}) -> {status}")

### 2.1 Máy tính wᵢⱼ cho một cặp ô bất kỳ

Chọn hai ô `i, j` (1–16), xem tích `ξᵢ·ξⱼ` trên từng mẫu rồi cộng lại thành `wᵢⱼ`.

In [ ]:
def hebb_pair_detail(i, j, patterns_dict, W):
    """In chi tiết phép tính w_ij từ ba mẫu (i, j: 1-based)."""
    ii, jj = i - 1, j - 1
    print(f"Cặp ô ({i}, {j}):")
    print(f"{'Mẫu':>5} | {'ξ_i':>4} | {'ξ_j':>4} | {'ξ_i·ξ_j':>8}")
    print("-" * 32)
    total = 0
    for name, p in patterns_dict.items():
        xi_i, xi_j = p[ii], p[jj]
        prod = xi_i * xi_j
        total += prod
        print(f"  {name}   | {int(xi_i):>+3}  | {int(xi_j):>+3}  |   {int(prod):>+3}")
    print("-" * 32)
    print(f"w_{i},{j} = {int(total):+d}   (kiểm tra với W: {int(W[ii, jj]):+d})")


# Thử các cặp trong phiếu bài tập
for (i, j) in [(2, 3), (4, 13), (8, 12)]:
    hebb_pair_detail(i, j, patterns, W)
    print()

---
## Phần 3: Hopfield khử nhiễu — vá từng pixel

**Luật cập nhật (bất đồng bộ):**
```
netᵢ = Σⱼ≠ᵢ wᵢⱼ·xⱼ
xᵢ ← sign(netᵢ)   (nếu net = 0 thì giữ nguyên)
```

Trên `x(0)` (C lật ô 4, 13), cả hai ô nhiễu có `net = 15 > 0` nên được kéo về đen (+1).

In [ ]:
def sign(val, current):
    """Hàm sign: +1 nếu > 0, -1 nếu < 0, giữ nguyên nếu = 0."""
    if val > 0:
        return 1.0
    elif val < 0:
        return -1.0
    else:
        return current


def energy(W, x):
    """E(x) = -1/2 * x^T W x"""
    return -0.5 * x @ W @ x


def hopfield_update(W, x, order=None, max_sweeps=10, verbose=True):
    """Cập nhật bất đồng bộ mạng Hopfield, in log từng bước."""
    x = x.copy().astype(float)
    n = len(x)
    if order is None:
        order = list(range(n))

    if verbose:
        print(f"Trạng thái ban đầu: E = {energy(W, x):.0f}")

    for sweep in range(max_sweeps):
        changed = False
        for i in order:
            net_i = W[i] @ x
            old_val = x[i]
            new_val = sign(net_i, old_val)
            if new_val != old_val:
                x[i] = new_val
                changed = True
                if verbose:
                    print(f"  Vòng {sweep+1}, ô {i+1}: net_{i+1} = {net_i:.0f} "
                          f"→ x_{i+1}: {int(old_val):+d} → {int(new_val):+d} "
                          f"| E = {energy(W, x):.0f}")
        if not changed:
            if verbose:
                print(f"\n✓ Ổn định sau {sweep+1} vòng quét. E cuối = {energy(W, x):.0f}")
            break
    return x


print("Khử nhiễu x(0) = chữ C lật ô 4 và ô 13\n")
x_result = hopfield_update(W, x0, order=list(range(16)))

print(f"\nKết quả: {np.array_equal(x_result, xiC)} (khớp mẫu C sạch)")

fig, axes = plt.subplots(1, 2, figsize=(6.4, 3.2))
show_grid(axes[0], x0, 'x(0) · nhiễu 2 ô')
show_grid(axes[1], x_result, 'x* · sau khi vá')
plt.tight_layout()
plt.show()

### 3.1 Kiểm tra: ba mẫu chuẩn có tự ổn định không?

Nếu W học đúng, đưa thẳng một mẫu chuẩn vào thì không ô nào nên đổi (net cùng dấu với x hiện tại).

In [ ]:
for name, p in patterns.items():
    out = hopfield_update(W, p, order=list(range(16)), verbose=False)
    ok = np.array_equal(out, p)
    print(f"Mẫu {name}: ổn định = {ok}")

### 3.2 Thử các preset nhiễu khác (I lật ô 8, T lật 3 ô)

Giống ba preset trong demo tương tác của bài giảng buổi 3.

In [ ]:
def make_noisy(letter, flip_indices_1based):
    """Tạo bản nhiễu của một mẫu bằng cách lật các ô (1-based)."""
    x = patterns[letter].copy()
    for idx in flip_indices_1based:
        x[idx - 1] *= -1
    return x


presets = {
    'C · lật ô 4 & 13': make_noisy('C', [4, 13]),
    'I · lật ô 8':       make_noisy('I', [8]),
    'T · lật 3 ô (1,5,6)': make_noisy('T', [1, 5, 6]),
}

fig, axes = plt.subplots(3, 2, figsize=(6.4, 9))
for row, (label, x_noisy) in enumerate(presets.items()):
    x_clean = hopfield_update(W, x_noisy, order=list(range(16)), verbose=False)
    show_grid(axes[row, 0], x_noisy, f'{label}\n(nhiễu)')
    show_grid(axes[row, 1], x_clean, 'sau khi vá')
    matched = [name for name, p in patterns.items() if np.array_equal(x_clean, p)]
    print(f"{label}: hội tụ về {matched if matched else 'trạng thái giả (không khớp mẫu nào)'}")
plt.tight_layout()
plt.show()

---
## Phần 4: Năng lượng E — vì sao mạng luôn dừng

**Công thức:** `E(x) = −½ xᵀWx = −½ Σᵢ xᵢ·netᵢ`

Mỗi lần vá một ô theo `sign(netᵢ)`, `E` **không bao giờ tăng** — chỉ giảm hoặc giữ nguyên. Vì lưới 16 ô chỉ có `2¹⁶ = 65536` trạng thái, mạng không thể lặp mãi và phải dừng (hội tụ).

In [ ]:
def hopfield_trace_energy(W, x, order=None, max_sweeps=10):
    """Chạy Hopfield và trả về danh sách (mô tả, E) sau mỗi lần đổi ô."""
    x = x.copy().astype(float)
    n = len(x)
    if order is None:
        order = list(range(n))

    trace = [("Bước 0 · x(0)", energy(W, x))]
    for sweep in range(max_sweeps):
        changed = False
        for i in order:
            net_i = W[i] @ x
            old_val = x[i]
            new_val = sign(net_i, old_val)
            if new_val != old_val:
                x[i] = new_val
                changed = True
                trace.append((f"Vá ô {i+1}", energy(W, x)))
        if not changed:
            break
    return x, trace


x_final, trace = hopfield_trace_energy(W, x0, order=list(range(16)))

print(f"{'Bước':>18} | {'E':>6}")
print("-" * 28)
for label, e in trace:
    print(f"{label:>18} | {e:>6.0f}")

steps = [t[0] for t in trace]
energies = [t[1] for t in trace]

plt.figure(figsize=(6, 3.5))
plt.plot(range(len(energies)), energies, marker='o')
plt.xticks(range(len(steps)), steps, rotation=30, ha='right', fontsize=8)
plt.ylabel('Năng lượng E')
plt.title('E giảm dần qua từng bước vá, không bao giờ tăng')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 4.1 Giới hạn dung lượng: ≈0,14N

Với `N=16`, nên lưu khoảng `0,14×16 ≈ 2` mẫu. Ba chữ C/I/T có cấu trúc (nhiều pixel giống nhau) nên vẫn chạy tốt, nhưng nhồi quá nhiều mẫu ngẫu nhiên sẽ dễ tạo **trạng thái giả** (spurious states).

In [ ]:
N = 16
capacity = 0.14 * N
print(f"N = {N} nơ-ron → dung lượng an toàn ước lượng ≈ 0,14N = {capacity:.2f} mẫu")
print("Bài này dùng 3 mẫu (C, I, T) — vượt mức lý thuyết nhưng vẫn chạy đúng nhờ")
print("cấu trúc đặc biệt (nhiều ô giống nhau) của ba chữ này, không phải mẫu ngẫu nhiên.")

---
## So sánh Hamming và Hopfield

| Tiêu chí | Hamming (buổi 2) | Hopfield (buổi 3) |
|----------|------------------|--------------------|
| **Input** | Ảnh nhiễu | Ảnh nhiễu |
| **Xử lý** | `yₖ = ξₖᵀx` (một lần nhân) | `xᵢ ← sign(netᵢ)` (lặp qua W) |
| **Output** | Nhãn "gần C" | Chữ C sạch (x*) |
| **Sửa ô sai?** | Không | Có, từng ô |
| **Kiến trúc** | So mẫu, một lượt | Hồi tiếp, cập nhật bất đồng bộ |

**Một câu để nhớ:** Hamming dán nhãn; Hopfield lặp `sign(Wx)` cho tới khi ảnh chạm đúng một chữ đã nhớ.

---
## Bài tập

**A.** Tính tay bốn trọng số Hebb: `w₁₄, w₂₃, w₅₆, w₁₂` (đáp án: 3, 3, −3, 1). Kiểm tra lại bằng `hebb_pair_detail`.

**B.** Với `x(0)` như bài (C lật ô 4, 13), viết `y_C, y_I, y_T` theo Hamming và một câu kết luận.

**C.** Lật ô 8 của chữ I, viết lưới `x(0)` và giải thích vì sao ô 8 thường bị kéo về −1.

**D.** Chạy `hopfield_update` cho preset "T lật 3 ô (1,5,6)"; dán log vòng đầu và vòng cuối, ghi số vòng quét tới khi hội tụ.

**E.** Viết 5–7 câu so sánh Hamming và Hopfield trên cùng bộ C/I/T.

**F.** (Mở rộng) Thử nhồi thêm một mẫu chữ thứ 4 vào W (ví dụ chữ L hoặc chữ H tự thiết kế trên lưới 4×4). Chạy lại Phần 3 và Phần 4 — mạng có còn hội tụ đúng cả 4 mẫu không? Vì sao?